# 02 — Train YOLO26n, Evaluate & Export for Raspberry Pi  ·  run on **T4 GPU**

`Runtime → Change runtime type → T4 GPU`. Needs `MyDrive/maize_weed_yolo26/maizeweed_dataset.zip` (the dataset made by notebook **01**).

Everything this notebook saves goes into **`MyDrive/maize_weed_yolo26/`**, with `yolo26n_v1` in the run and figure folder names, so results from different models never mix.

Saved to Drive:
- `runs/<RUN>/` — weights and Ultralytics' own plots (survives disconnects)
- `figures/<RUN>/` — **report figures** (300-dpi PNG + PDF), **tables** (CSV + LaTeX), `CAPTIONS.md`, `run_info.json`
- `maize_weed_yolo26n_ncnn_320.zip`, `_416.zip`, `_512.zip`, `_640.zip` — ready-to-use Pi models (NCNN) at each input size, each with `robot_config.json` (thresholds); 416 is the default

Final numbers for your report come from the **test** split, which the model never sees during training. A separate **Nigerian-only test** score shows performance on local conditions.

## 1. Setup

In [ ]:
!nvidia-smi -L
from google.colab import drive
drive.mount('/content/drive')
!pip -q install ultralytics

In [ ]:
import os, glob, json, random, shutil, time
DRIVE_DIR   = '/content/drive/MyDrive/maize_weed_yolo26'   # everything is saved here
DATASET_ZIP = f'{DRIVE_DIR}/maizeweed_dataset.zip'
OUT         = '/content/maizeweed'
RUN         = 'yolo26n_v1'        # new name for each experiment
MODEL       = 'yolo26n.pt'
PKG_PREFIX  = 'maize_weed_yolo26n_ncnn'   # Pi zips: <PKG_PREFIX>_<size>.zip
EPOCHS, PATIENCE, IMGSZ, BATCH, SEED = 100, 25, 640, 32, 0
ROBOT_SIZE   = 416                # input size used on the Pi
EXPORT_SIZES = [320, 416, 512, 640]
MAIZE_RECALL_TARGET = 0.95        # used to choose the maize-protection threshold
WEED_CONF_MIN = 0.25              # the robot never removes a plant below this confidence
MAIZE_CONF_MIN = 0.20             # floor for the maize threshold: lower values create many spurious protection boxes
FIG_DIR  = f'{DRIVE_DIR}/figures/{RUN}'
RUN_DIR  = f'{DRIVE_DIR}/runs/{RUN}'
os.makedirs(FIG_DIR, exist_ok=True)

if not os.path.exists(f'{OUT}/data.yaml'):
    if not os.path.exists(DATASET_ZIP):                               # help if the zip is somewhere else in Drive
        found = glob.glob('/content/drive/MyDrive/**/maizeweed_dataset*.zip', recursive=True)
        raise FileNotFoundError(f'Dataset zip not found at {DATASET_ZIP}. Found these instead: {found} - move it into {DRIVE_DIR}/ or change DATASET_ZIP.')
    !unzip -q "{DATASET_ZIP}" -d /content
open(f'{OUT}/data.yaml', 'w').write(f'path: {OUT}\ntrain: images/train\nval: images/val\ntest: images/test\nnames:\n  0: maize\n  1: weed\n')
ng_test = sorted(glob.glob(f'{OUT}/images/test/nigeria_*.jpg'))
open(f'{OUT}/test_nigeria.txt', 'w').write('\n'.join(ng_test))
open(f'{OUT}/data_nigeria.yaml', 'w').write(f'path: {OUT}\ntrain: images/train\nval: test_nigeria.txt\ntest: test_nigeria.txt\nnames:\n  0: maize\n  1: weed\n')
ng_val = sorted(glob.glob(f'{OUT}/images/val/nigeria_*.jpg'))      # used ONLY to choose the robot's thresholds
open(f'{OUT}/val_nigeria.txt', 'w').write('\n'.join(ng_val))
open(f'{OUT}/data_nigeria_val.yaml', 'w').write(f'path: {OUT}\ntrain: images/train\nval: val_nigeria.txt\nnames:\n  0: maize\n  1: weed\n')
counts = {s: len(os.listdir(f'{OUT}/images/{s}')) for s in ('train', 'val', 'test')}
print(counts, '| Nigerian val images:', len(ng_val), '| Nigerian test images:', len(ng_test))

In [ ]:
# ---------- Figure style (used for every report figure) ----------
import matplotlib as mpl, matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
INK, INK2, GRID, SURFACE = '#0b0b0b', '#52514e', '#e4e3df', '#fcfcfb'
CLS_COLOR = {'maize': '#008300', 'weed': '#4a3aa7'}           # validated colour-blind-safe pair
SERIES = ['#2a78d6', '#eb6834', '#1baf7a', '#eda100', '#e87ba4', '#008300', '#4a3aa7', '#e34948']  # fixed order
BOX_COLOR = {0: (0, 200, 83), 1: (255, 64, 214)}             # bright overlays on photos: maize green, weed magenta
SEQ_CMAP = LinearSegmentedColormap.from_list('seq_blue', ['#fcfcfb', '#cde2fb', '#86b6ef', '#2a78d6', '#184f95', '#0d366b'])
mpl.rcParams.update({
    'figure.dpi': 110, 'savefig.dpi': 300, 'figure.facecolor': SURFACE, 'axes.facecolor': SURFACE,
    'savefig.facecolor': SURFACE, 'font.size': 10, 'axes.titlesize': 11, 'axes.titleweight': 'bold',
    'axes.titlecolor': INK, 'axes.labelcolor': INK2, 'axes.edgecolor': GRID, 'axes.spines.top': False,
    'axes.spines.right': False, 'axes.grid': True, 'grid.color': GRID, 'grid.linewidth': 0.6,
    'axes.axisbelow': True, 'axes.grid.axis': 'y', 'xtick.color': INK2, 'ytick.color': INK2, 'legend.frameon': False,
    'lines.linewidth': 2, 'font.family': 'DejaVu Sans'})

def savefig(fig, name, show=True):
    os.makedirs(FIG_DIR, exist_ok=True)
    for ext in ('png', 'pdf'):
        fig.savefig(f'{FIG_DIR}/{name}.{ext}', bbox_inches='tight')
    print(f'saved {FIG_DIR}/{name}.png/.pdf')
    if show: plt.show()
    plt.close(fig)

def label_bars(ax, bars, fmt='{:,.0f}'):
    for b in bars:
        v = b.get_height()
        if v <= 0: continue
        ax.annotate(fmt.format(v), (b.get_x() + b.get_width() / 2, v), xytext=(0, 3),
                    textcoords='offset points', ha='center', va='bottom', fontsize=8, color=INK2)

def save_table(df, name):
    os.makedirs(FIG_DIR, exist_ok=True)
    df.to_csv(f'{FIG_DIR}/{name}.csv', index=False)
    try:
        with open(f'{FIG_DIR}/{name}.tex', 'w') as f: f.write(df.to_latex(index=False, float_format='%.3f'))
    except Exception: pass
    print(f'saved {FIG_DIR}/{name}.csv/.tex')

from PIL import ImageDraw
def draw_boxes(im, boxes, width=None, colors=None):
    """boxes: list of (cls, x0, y0, x1, y1) in pixels. Returns a copy with boxes drawn."""
    im = im.copy(); d = ImageDraw.Draw(im); width = width or max(2, round(max(im.size) / 250))
    for c, x0, y0, x1, y1 in boxes:
        d.rectangle([x0, y0, x1, y1], outline=(colors or BOX_COLOR)[int(c)], width=width)
    return im

def read_label(img_path):
    im = Image.open(img_path); W, H = im.size; out = []
    lp = img_path.replace('/images/', '/labels/').rsplit('.', 1)[0] + '.txt'
    if os.path.exists(lp):
        for line in open(lp):
            c, cx, cy, w, h = map(float, line.split())
            out.append((int(c), (cx - w/2) * W, (cy - h/2) * H, (cx + w/2) * W, (cy + h/2) * H))
    return im, out

from matplotlib.patches import Patch
BOX_LEGEND = [Patch(facecolor='none', edgecolor=tuple(v/255 for v in BOX_COLOR[0]), linewidth=2, label='maize'),
              Patch(facecolor='none', edgecolor=tuple(v/255 for v in BOX_COLOR[1]), linewidth=2, label='weed')]

import numpy as np, pandas as pd
from collections import Counter
from PIL import Image
CLASS_NAMES = ['maize', 'weed']
SUBSET_COLOR = {'All test images': SERIES[0], 'Nigerian test images': SERIES[1]}
captions = {}
def caption(name, text):
    captions[name] = text
    with open(f'{FIG_DIR}/CAPTIONS.md', 'w') as f:
        for k, v in captions.items(): f.write(f'**{k}** — {v}\n\n')

## 2. Train

In [ ]:
import ultralytics, torch, platform
from ultralytics import YOLO
info = {'run': RUN, 'model': MODEL, 'epochs': EPOCHS, 'patience': PATIENCE, 'imgsz': IMGSZ, 'batch': BATCH, 'seed': SEED,
        'augment': {'flipud': 0.5, 'degrees': 10}, 'dataset_images': counts, 'nigeria_test_images': len(ng_test),
        'ultralytics': ultralytics.__version__, 'torch': torch.__version__, 'python': platform.python_version(),
        'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none'}
json.dump(info, open(f'{FIG_DIR}/run_info.json', 'w'), indent=1); print(info)

model = YOLO(MODEL)
t0 = time.time()
model.train(data=f'{OUT}/data.yaml', epochs=EPOCHS, patience=PATIENCE, imgsz=IMGSZ, batch=BATCH, seed=SEED,
            flipud=0.5, degrees=10,          # top-down camera: vertical flips and small rotations are realistic
            project=f'{DRIVE_DIR}/runs', name=RUN, exist_ok=True)
info['train_hours'] = round((time.time() - t0) / 3600, 2); json.dump(info, open(f'{FIG_DIR}/run_info.json', 'w'), indent=1)

In [ ]:
# Resume after a disconnect (run Section 1 first)
from ultralytics import YOLO
YOLO(f'{RUN_DIR}/weights/last.pt').train(resume=True)

## 3. Training curves
**If Colab disconnected after training finished** (e.g. GPU time ran out): switch to a CPU runtime, run the three cells of Section 1, **skip** the two Section 2 cells, then click the cell below and use *Runtime → Run cell and below*. Everything from here works on CPU (≈30–60 min).

In [ ]:
res = pd.read_csv(f'{RUN_DIR}/results.csv'); res.columns = [c.strip() for c in res.columns]
fit = 0.1 * res['metrics/mAP50(B)'] + 0.9 * res['metrics/mAP50-95(B)']; best_ep = int(res.loc[fit.idxmax(), 'epoch'])
fig, axes = plt.subplots(2, 4, figsize=(15, 6.5))
LOSS_TITLE = {'box_loss': 'Box loss', 'cls_loss': 'Class loss', 'dfl_loss': 'DFL loss', 'l1_loss': 'L1 box loss'}
loss_keys = [k for k in ('box_loss', 'cls_loss', 'dfl_loss', 'l1_loss') if f'train/{k}' in res.columns][:3]   # YOLO11: dfl, YOLO26: l1
for ax, k in zip(axes[0], loss_keys):
    title = LOSS_TITLE[k]
    ax.plot(res.epoch, res[f'train/{k}'], color=SERIES[0], label='train')
    ax.plot(res.epoch, res[f'val/{k}'], color=SERIES[1], label='validation')
    ax.set_title(title); ax.set_xlabel('epoch')
axes[0, 0].legend()
axes[0, 3].plot(res.epoch, res['lr/pg0'], color=SERIES[0]); axes[0, 3].set_title('Learning rate'); axes[0, 3].set_xlabel('epoch')
for ax, k, title in zip(axes[1], ['precision(B)', 'recall(B)', 'mAP50(B)', 'mAP50-95(B)'], ['Precision', 'Recall', 'mAP@0.5', 'mAP@0.5:0.95']):
    ax.plot(res.epoch, res[f'metrics/{k}'], color=SERIES[0]); ax.set_title(f'{title} (validation)'); ax.set_xlabel('epoch'); ax.set_ylim(0, 1)
from matplotlib.ticker import MaxNLocator
for ax in axes.flat:
    ax.xaxis.set_major_locator(MaxNLocator(integer=True))
    ax.axvline(best_ep, color=INK2, linestyle='--', linewidth=1)
axes[1, 3].annotate(f'best epoch {best_ep}', (best_ep, 0.05), xytext=(4, 0), textcoords='offset points', fontsize=8, color=INK2)
plt.tight_layout(); savefig(fig, 'fig8_training_curves')
caption('fig8_training_curves', f'Training and validation losses, learning rate and validation metrics per epoch. Dashed line: selected (best) epoch {best_ep}.')

## 4. Test-set evaluation
Two subsets: **all test images** and **Nigerian test images only**.

In [ ]:
BEST = f'{RUN_DIR}/weights/best.pt'
model = YOLO(BEST)
def count_instances(img_list):
    n = np.zeros(2, int)
    for ip in img_list:
        lp = ip.replace('/images/', '/labels/')[:-4] + '.txt'
        if os.path.exists(lp):
            for line in open(lp): n[int(line.split()[0])] += 1
    return n
SUBSETS = {'All test images': (f'{OUT}/data.yaml', sorted(glob.glob(f'{OUT}/images/test/*.jpg'))),
           'Nigerian test images': (f'{OUT}/data_nigeria.yaml', ng_test)}
METRICS, rows = {}, []
for name, (yml, files) in SUBSETS.items():
    m = model.val(data=yml, split='test', imgsz=IMGSZ, batch=BATCH, plots=True, verbose=False,
                  project=f'{FIG_DIR}/ultralytics_val', name=name.split()[0].lower(), exist_ok=True)
    METRICS[name] = m; inst = count_instances(files)
    rows.append({'subset': name, 'class': 'all', 'images': len(files), 'instances': int(inst.sum()),
                 'precision': m.box.mp, 'recall': m.box.mr, 'mAP50': m.box.map50, 'mAP50-95': m.box.map})
    for i, c in enumerate(m.box.ap_class_index):
        p, r, ap50, ap = m.box.class_result(i)
        rows.append({'subset': name, 'class': CLASS_NAMES[int(c)], 'images': len(files), 'instances': int(inst[int(c)]),
                     'precision': p, 'recall': r, 'mAP50': ap50, 'mAP50-95': ap})
test_tab = pd.DataFrame(rows); save_table(test_tab.round(4), 'table4_test_metrics')
caption('table4_test_metrics', 'Detection performance on the held-out test set (all sources) and on the Nigerian test images only.')
display(test_tab.round(3))

In [ ]:
# Fig 9 - per-class metrics: all test images vs Nigerian test images
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8), sharey=True)
mets = ['precision', 'recall', 'mAP50', 'mAP50-95']; x = np.arange(len(mets)); wd = 0.38
for ax, c in zip(axes, CLASS_NAMES):
    for i, (sub, col) in enumerate(SUBSET_COLOR.items()):
        r = test_tab[(test_tab.subset == sub) & (test_tab['class'] == c)]
        if len(r): label_bars(ax, ax.bar(x + (i - .5) * wd, r[mets].values[0], wd, color=col, label=sub, edgecolor=SURFACE), fmt='{:.2f}')
    ax.set_xticks(x, ['Precision', 'Recall', 'mAP@0.5', 'mAP@0.5:0.95']); ax.set_ylim(0, 1.08); ax.set_title(c.capitalize())
axes[0].set_ylabel('score')
fig.legend(*axes[0].get_legend_handles_labels(), loc='upper center', ncol=2, bbox_to_anchor=(0.5, 1.06), fontsize=9)
savefig(fig, 'fig9_per_class_metrics')
caption('fig9_per_class_metrics', 'Per-class precision, recall and mAP on all test images versus the Nigerian test images.')

In [ ]:
# Fig 10-11 - precision-recall and F1-confidence curves (fallback: Ultralytics' own plots)
def curves(m):
    try: return m.box.curves_results       # [PR, F1-conf, P-conf, R-conf]
    except Exception: return None
fig_pr, ax_pr = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
fig_f1, ax_f1 = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
ok = True
for j, (sub, m) in enumerate(METRICS.items()):
    cr = curves(m)
    if cr is None: ok = False; break
    (px, prec, *_), (cx, f1, *_) = cr[0], cr[1]
    for i, c in enumerate(m.box.ap_class_index):
        name = CLASS_NAMES[int(c)]; ap50 = m.box.class_result(i)[2]
        ax_pr[j].plot(px, prec[i], color=CLS_COLOR[name], label=f'{name}  AP@0.5 = {ap50:.3f}')
        k = int(np.argmax(f1[i])); ax_f1[j].plot(cx, f1[i], color=CLS_COLOR[name], label=f'{name}  best F1 {f1[i][k]:.2f} @ conf {cx[k]:.2f}')
        ax_f1[j].plot(cx[k], f1[i][k], 'o', color=CLS_COLOR[name], markersize=7, markeredgecolor=SURFACE)
    for ax, xl, yl in ((ax_pr[j], 'Recall', 'Precision'), (ax_f1[j], 'Confidence threshold', 'F1')):
        ax.set_title(sub); ax.set_xlabel(xl); ax.set_xlim(0, 1); ax.set_ylim(0, 1.02); ax.legend(loc='lower left', fontsize=8)
    ax_pr[j].set_ylabel('Precision') if j == 0 else None; ax_f1[j].set_ylabel('F1') if j == 0 else None
if ok:
    savefig(fig_pr, 'fig10_precision_recall'); savefig(fig_f1, 'fig11_f1_confidence')
else:
    plt.close(fig_pr); plt.close(fig_f1)
    for sub, m in METRICS.items():
        for f in ('PR_curve', 'BoxPR_curve', 'F1_curve', 'BoxF1_curve'):
            p = f'{m.save_dir}/{f}.png'
            if os.path.exists(p): shutil.copy(p, f'{FIG_DIR}/ultralytics_{sub.split()[0].lower()}_{f}.png')
    print('Used Ultralytics PR/F1 plots instead (see ultralytics_*.png).')
caption('fig10_precision_recall', 'Precision-recall curves per class on all test images (left) and Nigerian test images (right).')
caption('fig11_f1_confidence', 'F1 score versus confidence threshold; dots mark the best threshold per class.')

In [ ]:
# Fig 12 - normalised confusion matrices (columns = true class)
fig, axes = plt.subplots(1, 2, figsize=(10, 4.2))
labels = ['maize', 'weed', 'background']
for ax, (sub, m) in zip(axes, METRICS.items()):
    try: M = np.array(m.confusion_matrix.matrix, dtype=float)      # rows = predicted, cols = true
    except Exception: M = None
    if M is None or M.shape != (3, 3): ax.text(.5, .5, 'not available', ha='center'); ax.axis('off'); continue
    N = M / np.maximum(M.sum(0, keepdims=True), 1)
    ax.imshow(N, cmap=SEQ_CMAP, vmin=0, vmax=1); ax.grid(False)
    for (r, c), v in np.ndenumerate(N):
        if r == 2 and c == 2: continue
        ax.text(c, r, f'{v:.2f}', ha='center', va='center', fontsize=10, color='white' if v > 0.55 else INK)
    ax.set_xticks(range(3), labels); ax.set_yticks(range(3), labels)
    ax.set_xlabel('true class'); ax.set_ylabel('predicted class'); ax.set_title(sub)
plt.tight_layout(); savefig(fig, 'fig12_confusion_matrix')
caption('fig12_confusion_matrix', 'Normalised confusion matrices (each column sums to 1). "Background" = missed objects (column) or false detections (row).')

## 5. Choose the robot's thresholds (on the **validation** set)
The robot uses **two** confidence thresholds. They are chosen on the **Nigerian validation images** (never used for final scoring), so the test-set numbers in Tables 4–5 stay unbiased:
- **maize threshold (low)** — anything above it is protected. The highest threshold that still finds `MAIZE_RECALL_TARGET` (95 %) of maize, but never below `MAIZE_CONF_MIN` (0.20) — lower values add many spurious maize boxes that would block weeding.
- **weed threshold (high)** — only weeds above it are removed. Chosen at the best weed F1, never below `WEED_CONF_MIN` (0.25).

In [ ]:
th_name = 'Nigerian validation images' if ng_val else 'all validation images'
m = model.val(data=f'{OUT}/data_nigeria_val.yaml' if ng_val else f'{OUT}/data.yaml', split='val', imgsz=IMGSZ, batch=BATCH,
              plots=False, verbose=False, project=f'{FIG_DIR}/ultralytics_val', name='threshold_val', exist_ok=True)
cr = curves(m); cls_idx = {CLASS_NAMES[int(c)]: i for i, c in enumerate(m.box.ap_class_index)}
if cr:
    cx = cr[2][0]; P, R, F1 = cr[2][1], cr[3][1], cr[1][1]
    mi, wi = cls_idx['maize'], cls_idx['weed']
    ok_idx = np.where(R[mi] >= MAIZE_RECALL_TARGET)[0]
    MAIZE_CONF = max(float(cx[ok_idx.max()]) if len(ok_idx) else 0.05, MAIZE_CONF_MIN)
    WEED_CONF = max(float(cx[int(np.argmax(F1[wi]))]), WEED_CONF_MIN)   # never act on very unsure weeds
    k = int(np.argmin(np.abs(cx - MAIZE_CONF)))
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(cx, R[mi], color=CLS_COLOR['maize'], label='maize recall (share of maize protected)')
    ax.plot(cx, P[wi], color=CLS_COLOR['weed'], label='weed precision (share of removals that are real weeds)')
    ax.plot(cx, R[wi], color=CLS_COLOR['weed'], linestyle='--', label='weed recall (share of weeds removed)')
    ax.axvline(MAIZE_CONF, color=CLS_COLOR['maize'], linestyle=':', linewidth=1.5); ax.axvline(WEED_CONF, color=CLS_COLOR['weed'], linestyle=':', linewidth=1.5)
    m_left = MAIZE_CONF <= WEED_CONF                                   # put each label on the outer side so they never overlap
    ax.text(MAIZE_CONF, 0.04, f' maize {MAIZE_CONF:.2f} ' if not m_left else f'maize {MAIZE_CONF:.2f} ', transform=ax.get_xaxis_transform(), fontsize=8, color=INK2, ha='right' if m_left else 'left')
    ax.text(WEED_CONF, 0.04, f' weed {WEED_CONF:.2f}' if m_left else f'weed {WEED_CONF:.2f} ', transform=ax.get_xaxis_transform(), fontsize=8, color=INK2, ha='left' if m_left else 'right')
    ax.set_xlim(0, 1); ax.set_ylim(0, 1.02); ax.set_xlabel('confidence threshold'); ax.set_ylabel('score')
    ax.set_title(f'Threshold selection for the weeding robot ({th_name})'); ax.legend(fontsize=8, loc='upper center', bbox_to_anchor=(0.5, -0.14), ncol=1)
    savefig(fig, 'fig13_threshold_selection')
    caption('fig13_threshold_selection', f'Maize recall, weed precision and weed recall versus confidence threshold on the {th_name} ({len(ng_val) if ng_val else counts["val"]} images; not used for final scoring). '
            f'Chosen thresholds: maize {MAIZE_CONF:.2f} (maize recall {R[mi][k]:.0%}; target {MAIZE_RECALL_TARGET:.0%}, minimum {MAIZE_CONF_MIN}), weed {WEED_CONF:.2f} (best weed F1, minimum {WEED_CONF_MIN}).')
else:
    MAIZE_CONF, WEED_CONF = 0.20, 0.40
print(f'Chosen on {th_name}:  MAIZE_CONF = {MAIZE_CONF:.2f}   WEED_CONF = {WEED_CONF:.2f}')

## 6. Safety check at the chosen thresholds
Counts the errors that matter for a weeding robot — especially **maize predicted as weed** (the robot would pull a maize plant).

In [ ]:
def iou(a, b):
    ix = max(0, min(a[2], b[2]) - max(a[0], b[0])); iy = max(0, min(a[3], b[3]) - max(a[1], b[1])); inter = ix * iy
    return inter / ((a[2]-a[0]) * (a[3]-a[1]) + (b[2]-b[0]) * (b[3]-b[1]) - inter + 1e-9)

def analyse(files, batch=32):
    per_img, tot = [], Counter()
    for k in range(0, len(files), batch):
        for fp, r in zip(files[k:k+batch], model.predict(files[k:k+batch], imgsz=IMGSZ, conf=min(MAIZE_CONF, WEED_CONF), verbose=False)):
            _, gt = read_label(fp)
            pr = [(int(c), *b) for c, b, s in zip(r.boxes.cls.tolist(), r.boxes.xyxy.tolist(), r.boxes.conf.tolist())
                  if s >= (MAIZE_CONF if int(c) == 0 else WEED_CONF)]
            e = Counter(); missed, killed = [], []
            for g in gt:
                best = max(((iou(g[1:], p[1:]), p[0]) for p in pr), default=(0, -1))
                same = any(iou(g[1:], p[1:]) >= 0.5 and p[0] == g[0] for p in pr)
                if g[0] == 0:
                    e['maize_gt'] += 1; e['maize_found'] += same
                    if not same: missed.append(g)
                    if any(iou(g[1:], p[1:]) >= 0.5 and p[0] == 1 for p in pr) and not same: e['maize_called_weed'] += 1; killed.append(g)
                else:
                    e['weed_gt'] += 1; e['weed_found'] += same
            for p in pr:
                if p[0] == 1 and not any(iou(p[1:], g[1:]) >= 0.5 for g in gt): e['weed_false_alarm'] += 1
            tot.update(e); per_img.append((fp, e, gt, pr, missed, killed))
    return per_img, tot

SAFETY, rows = {}, []
for sub, (_, files) in SUBSETS.items():
    per_img, t = analyse(files); SAFETY[sub] = per_img
    rows.append({'subset': sub, 'maize plants': t['maize_gt'], 'maize protected %': 100 * t['maize_found'] / max(t['maize_gt'], 1),
                 'maize called weed %': 100 * t['maize_called_weed'] / max(t['maize_gt'], 1),
                 'weeds': t['weed_gt'], 'weeds removed %': 100 * t['weed_found'] / max(t['weed_gt'], 1),
                 'false weed detections': t['weed_false_alarm']})
safety_tab = pd.DataFrame(rows).round(2); save_table(safety_tab, 'table5_robot_safety')
caption('table5_robot_safety', f'Robot-level outcome on the test images at maize conf {MAIZE_CONF:.2f} and weed conf {WEED_CONF:.2f} (IoU ≥ 0.5). "Maize called weed" are plants the robot would wrongly remove.')
display(safety_tab)

In [ ]:
# Fig 14 - ground truth vs prediction: 2 Nigerian + 2 other-source test images
SRC_NAME = {'nigeria': 'Nigeria', 'cornweed': 'CornWeed', 'cropandweed': 'CropAndWeed', 'acre': 'ACRE', 'rf_cornweed': 'Roboflow corn-weed', 'weedmaize': 'WeedMaize'}
def src_of(fp): return os.path.basename(fp).rsplit('_', 1)[0]
random.seed(3)
pool = SAFETY['All test images']
ng_pool = [t for t in pool if src_of(t[0]) == 'nigeria']; other = [t for t in pool if src_of(t[0]) != 'nigeria']
pick = random.sample(ng_pool, min(2, len(ng_pool))) + random.sample(other, min(4 - min(2, len(ng_pool)), len(other)))
fig, axes = plt.subplots(len(pick), 2, figsize=(11, 3.4 * len(pick)), squeeze=False)
for (fp, e, gt, pr, *_), (a, b) in zip(pick, axes):
    im = Image.open(fp); nm = SRC_NAME.get(src_of(fp), src_of(fp))
    a.imshow(draw_boxes(im, gt)); a.set_title(f'Ground truth · {nm}', loc='left', fontsize=9); a.axis('off')
    b.imshow(draw_boxes(im, pr)); b.set_title('Prediction', loc='left', fontsize=9); b.axis('off')
fig.legend(handles=BOX_LEGEND, loc='upper right', ncol=2); plt.tight_layout(); savefig(fig, 'fig14_predictions_vs_ground_truth')
caption('fig14_predictions_vs_ground_truth', f'Ground truth (left) and model predictions at the robot thresholds (right) on test images from Nigeria and other sources (maize conf {MAIZE_CONF:.2f}, weed conf {WEED_CONF:.2f}). Green: maize, magenta: weed.')

# Fig 15 - hardest cases: maize missed (orange) or called weed (red)
errs = [t for t in sorted(pool, key=lambda t: (t[1]['maize_called_weed'], t[1]['maize_gt'] - t[1]['maize_found']), reverse=True)
        if t[1]['maize_called_weed'] or t[1]['maize_gt'] > t[1]['maize_found']]
worst, per_src = [], Counter()
for t in errs:                                   # at most 2 per source, so the figure shows different conditions
    if per_src[src_of(t[0])] < 2: worst.append(t); per_src[src_of(t[0])] += 1
    if len(worst) == 4: break
if not worst: print('No maize errors on these test images - nothing to show.'); worst = pool[:1]
fig, axes = plt.subplots(1, len(worst), figsize=(4.2 * len(worst), 3.6), squeeze=False)
for (fp, e, gt, pr, missed, killed), ax in zip(worst, axes[0]):
    im = draw_boxes(Image.open(fp), pr)
    im = draw_boxes(im, [(0, *g[1:]) for g in missed], colors={0: (237, 161, 0)})
    im = draw_boxes(im, [(0, *g[1:]) for g in killed], colors={0: (227, 73, 72)})
    ax.imshow(im); ax.axis('off'); ax.set_title(f"{SRC_NAME.get(src_of(fp), src_of(fp))} · missed {e['maize_gt']-e['maize_found']} · called weed {e['maize_called_weed']}", fontsize=9)
fig.legend(handles=BOX_LEGEND + [Patch(facecolor='none', edgecolor='#eda100', linewidth=2, label='maize missed'),
                                 Patch(facecolor='none', edgecolor='#e34948', linewidth=2, label='maize called weed')], loc='upper right', ncol=4, fontsize=8)
plt.tight_layout(); savefig(fig, 'fig15_failure_cases')
caption('fig15_failure_cases', 'Hardest test images (all sources): predictions with missed maize (orange) and maize predicted as weed (red) highlighted.')

## 7. Export for Raspberry Pi & speed/accuracy trade-off
Exports **NCNN** models at several input sizes and measures accuracy on **all test images** (main number) and on the **Nigerian test images**, plus speed.
Speed is measured on **Colab's Intel CPU**, so it is only for *comparing sizes*: NCNN is optimised for ARM and is much faster than PyTorch on the Raspberry Pi — measure real Pi times with the Pi script. Takes ~15–25 min on CPU.

In [ ]:
import torch
cpu_threads = torch.get_num_threads()
random.seed(0); bench_imgs = random.sample(SUBSETS['All test images'][1], min(40, len(SUBSETS['All test images'][1])))   # mixed sources
def bench(m, sz):
    for f in bench_imgs[:5]: m.predict(f, imgsz=sz, device='cpu', verbose=False)          # warm-up
    t = [m.predict(f, imgsz=sz, device='cpu', verbose=False)[0].speed['inference'] for f in bench_imgs]
    return float(np.median(t))
def acc(m, sz):
    out = {}
    for key, yml in (('all', f'{OUT}/data.yaml'), ('nigeria', f'{OUT}/data_nigeria.yaml')):
        if key == 'nigeria' and not ng_test: continue
        v = m.val(data=yml, split='test', imgsz=sz, batch=1, device='cpu', verbose=False, plots=False)
        out[f'mAP50_{key}'] = v.box.map50; out[f'mAP50-95_{key}'] = v.box.map
    return out
rows = []
pt = YOLO(BEST)
rows.append({'format': 'PyTorch', 'imgsz': ROBOT_SIZE, 'size_MB': os.path.getsize(BEST) / 1e6, 'latency_ms': bench(pt, ROBOT_SIZE), **acc(pt, ROBOT_SIZE)})
for sz in EXPORT_SIZES:
    d = YOLO(BEST).export(format='ncnn', imgsz=sz)
    dst = f'{RUN_DIR}/weights/best_{sz}_ncnn_model'   # folder name must end in _ncnn_model
    shutil.rmtree(dst, ignore_errors=True); shutil.move(d, dst)
    m = YOLO(dst, task='detect')
    mb = sum(os.path.getsize(p) for p in glob.glob(f'{dst}/*')) / 1e6
    rows.append({'format': 'NCNN', 'imgsz': sz, 'size_MB': mb, 'latency_ms': bench(m, sz), **acc(m, sz)})
    print(rows[-1])
bench_tab = pd.DataFrame(rows); bench_tab['FPS'] = 1000 / bench_tab.latency_ms
save_table(bench_tab.round(3), 'table6_export_benchmark'); display(bench_tab.round(3))
caption('table6_export_benchmark', f'Model size, median inference latency on 40 mixed test images (Colab Intel CPU, {cpu_threads} thread(s); relative only) and mAP on all {counts["test"]} test images and the {len(ng_test)} Nigerian test images, per format and input size.')

fig, ax = plt.subplots(figsize=(8, 4.6))
nc, p = bench_tab[bench_tab.format == 'NCNN'], bench_tab[bench_tab.format == 'PyTorch'].iloc[0]
for key, lab, col in (('all', f'all test images ({counts["test"]})', SERIES[0]), ('nigeria', f'Nigerian test images ({len(ng_test)})', SERIES[1])):
    if f'mAP50_{key}' not in bench_tab: continue
    ax.plot(nc.latency_ms, nc[f'mAP50_{key}'], '-o', color=col, markersize=8, markeredgecolor=SURFACE, label=f'NCNN · {lab}')
    ax.plot(p.latency_ms, p[f'mAP50_{key}'], 's', color=col, markersize=9, markeredgecolor=SURFACE, label=f'PyTorch {int(p.imgsz)} px · {lab}')
for _, r in nc.iterrows(): ax.annotate(f"{int(r.imgsz)} px", (r.latency_ms, r['mAP50_all']), xytext=(6, 6), textcoords='offset points', fontsize=8, color=INK2)
ax.set_xlabel(f'median latency per image, ms (Colab Intel CPU, {cpu_threads} thread(s))'); ax.set_ylabel('mAP@0.5')
ax.set_title('Speed vs accuracy by input size'); ax.legend(fontsize=8, loc='upper center', bbox_to_anchor=(0.5, -0.16), ncol=2)
savefig(fig, 'fig16_speed_accuracy')
caption('fig16_speed_accuracy', 'Accuracy (all test images and Nigerian test images) versus CPU latency for NCNN exports at different input sizes, with PyTorch for reference. Latency measured on an Intel CPU is relative only; NCNN is the fastest format on ARM devices such as the Raspberry Pi.')

In [ ]:
# Package every exported size as a ready-to-use Pi zip (model + thresholds)
for sz in EXPORT_SIZES:
    src = f'{RUN_DIR}/weights/best_{sz}_ncnn_model'
    if not os.path.exists(src): print('missing', src); continue
    pkg = f'/content/{PKG_PREFIX}_{sz}'
    shutil.rmtree(pkg, ignore_errors=True); shutil.copytree(src, f'{pkg}/maize_weed_ncnn_model')
    json.dump({'imgsz': sz, 'classes': CLASS_NAMES, 'maize_conf': round(MAIZE_CONF, 3), 'weed_conf': round(WEED_CONF, 3),
               'model': MODEL, 'run': RUN}, open(f'{pkg}/robot_config.json', 'w'), indent=1)
    zp = f'{DRIVE_DIR}/{PKG_PREFIX}_{sz}.zip'
    if os.path.exists(zp): os.remove(zp)
    shutil.make_archive(zp[:-4], 'zip', '/content', f'{PKG_PREFIX}_{sz}')
    print(f'saved {zp}  ({os.path.getsize(zp)/1e6:.1f} MB)')
print(f'thresholds: maize {MAIZE_CONF:.2f}, weed {WEED_CONF:.2f}')
print('\nReport files:')
for f in sorted(glob.glob(f'{FIG_DIR}/*.*')): print(' ', os.path.basename(f))

## On the Raspberry Pi (64-bit Raspberry Pi OS)
```bash
pip install ultralytics ncnn
unzip maize_weed_yolo26n_ncnn_416.zip
```
```python
import json
from ultralytics import YOLO
cfg = json.load(open('maize_weed_yolo26n_ncnn_416/robot_config.json'))
model = YOLO('maize_weed_yolo26n_ncnn_416/maize_weed_ncnn_model', task='detect')
r = model.predict('test.jpg', imgsz=cfg['imgsz'], conf=cfg['maize_conf'])[0]
```